# Qwen2.5-**1.5B** QLoRA - CSRC RAG Fine-tuning (Colab T4 / Kaggle P100)

**项目**: Deeplearning_project-CSRC_Rag · 赛道 B · 证监会违规 RAG 问答系统

**为什么有这个 notebook**: 本地 RTX 2060 SUPER 8 GB 容不下 Qwen2.5-**1.5B** 的 QLoRA + max_seq=2048 (OOM at step 8)。本地训练已降级到 0.5B (见 `artifacts/models/qwen_lora_csrc/`)。这个 notebook 是 1.5B 的**备选**训练方案,预期在 T4 (16 GB) 上 3-5 小时完成 5500 样本 × 3 epoch。

### 使用顺序
1. 把本仓库 push 到 GitHub
2. Colab → New Notebook → 挂 Google Drive,把本仓库 clone 到 `/content/drive/MyDrive/csrc_rag`
3. 运行时类型 → T4 GPU
4. 从头跑完
5. 训练结束后 adapter 自动回写到 Drive 的 `artifacts/models/qwen_lora_csrc_1_5b/`,pull 回本地即可

> **和 0.5B notebook 的差异**: base_model 改 1.5B + max_seq=2048 + batch=1 + grad_accum=16,其他不变。

## 0. 环境检测 + Drive 挂载

In [ ]:
import os, sys, pathlib
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/input')
print('colab=', IS_COLAB, 'kaggle=', IS_KAGGLE)

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = pathlib.Path('/content/drive/MyDrive/csrc_rag')
elif IS_KAGGLE:
    WORK = pathlib.Path('/kaggle/working/csrc_rag')
else:
    WORK = pathlib.Path.cwd()
print('WORK =', WORK)
assert WORK.exists(), f'请先把项目 clone 到 {WORK}'

# 显卡
!nvidia-smi | head -20

## 1. Install pinned dependencies

T4 上经过验证的固定版本组合。不要盲目升级。

In [ ]:
%pip install -q \
    transformers==4.44.2 \
    peft==0.13.2 \
    bitsandbytes==0.43.3 \
    accelerate==0.33.0 \
    datasets==2.21.0 \
    trl==0.9.6 \
    sentencepiece

## 2. 配置 HuggingFace 镜像 (可选, 中国网络提速)

In [ ]:
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

## 3. 读配置 + 覆写为 1.5B 参数

复用 `configs/qlora_config.json`, 只覆写 base_model / max_seq / batch。

In [ ]:
import json, copy
CFG = json.loads((WORK/'configs'/'qlora_config.json').read_text(encoding='utf-8'))

# ---- 1.5B Colab 覆写 ----
CFG['meta']['base_model'] = 'Qwen/Qwen2.5-1.5B-Instruct'
CFG['training']['max_seq_length'] = 2048
CFG['training']['per_device_train_batch_size'] = 1
CFG['training']['per_device_eval_batch_size'] = 1
CFG['training']['gradient_accumulation_steps'] = 16
CFG['training']['num_train_epochs'] = 3
# T4 支持 fp16 (no bf16)。如果在 A100/L4 上可开 bf16。
CFG['training']['fp16'] = True
CFG['training']['bf16'] = False
# 输出目录 (独立于 0.5B)
CFG['training']['output_dir'] = 'artifacts/models/qwen_lora_csrc_1_5b'

print('base_model =', CFG['meta']['base_model'])
print('max_seq_length =', CFG['training']['max_seq_length'])
print('effective batch =', CFG['training']['per_device_train_batch_size'] * CFG['training']['gradient_accumulation_steps'])
print('output_dir =', CFG['training']['output_dir'])

train_path = WORK / 'data' / 'train' / 'rag_qa_train.jsonl'
val_path   = WORK / 'data' / 'train' / 'rag_qa_val.jsonl'
test_path  = WORK / 'data' / 'train' / 'rag_qa_test.jsonl'
assert train_path.exists(), f'缺数据: {train_path}'
print('train:', train_path.stat().st_size/1e6, 'MB')

## 4. 4-bit 加载 base model (NF4 + double quant)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL = CFG['meta']['base_model']

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=CFG['lora']['r'],
    lora_alpha=CFG['lora']['lora_alpha'],
    lora_dropout=CFG['lora']['lora_dropout'],
    bias=CFG['lora']['bias'],
    task_type=CFG['lora']['task_type'],
    target_modules=CFG['lora']['target_modules'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 5. Tokenize — Alpaca 格式转 chat messages

训练脚本 `scripts/train_qlora_m4.py` 已经做了这个转换,Colab 版这里内联一份。

In [ ]:
from datasets import Dataset
MAX_LEN = CFG['training']['max_seq_length']
SYS = CFG['data']['system_prompt']

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def to_messages(row):
    # 兼容两种结构: 已经是 messages 格式 / Alpaca (instruction+input+output)
    if 'messages' in row:
        return row['messages']
    instr = row.get('instruction', '')
    inp = row.get('input', '')
    out = row.get('output', '')
    user = instr + ('\n\n' + inp if inp else '')
    return [
        {'role': 'system', 'content': SYS},
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': out},
    ]

def encode(row):
    msgs = to_messages(row)
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    enc = tok(text, max_length=MAX_LEN, truncation=True, padding=False)
    enc['labels'] = enc['input_ids'].copy()
    return enc

train_rows = load_jsonl(train_path)
val_rows   = load_jsonl(val_path)
print('train rows:', len(train_rows), ' val rows:', len(val_rows))

train_ds = Dataset.from_list(train_rows).map(encode, remove_columns=list(train_rows[0].keys()))
val_ds   = Dataset.from_list(val_rows).map(encode, remove_columns=list(val_rows[0].keys()))
print('train tokens (first 3 rows lens):', [len(x['input_ids']) for x in train_ds.select(range(3))])

## 6. Trainer

用 `DataCollatorForSeq2Seq` (而非 `ForLanguageModeling`) — 后者会 shift labels, 对 causal LM + chat template 训练不对。

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback
tr = CFG['training']
OUT = WORK / tr['output_dir']
OUT.mkdir(parents=True, exist_ok=True)

args = TrainingArguments(
    output_dir=str(OUT),
    per_device_train_batch_size=tr['per_device_train_batch_size'],
    per_device_eval_batch_size=tr['per_device_eval_batch_size'],
    gradient_accumulation_steps=tr['gradient_accumulation_steps'],
    learning_rate=tr['learning_rate'],
    num_train_epochs=tr['num_train_epochs'],
    warmup_ratio=tr['warmup_ratio'],
    lr_scheduler_type=tr['lr_scheduler_type'],
    weight_decay=tr['weight_decay'],
    optim=tr['optim'],
    gradient_checkpointing=tr['gradient_checkpointing'],
    logging_steps=tr['logging_steps'],
    eval_strategy='steps',
    eval_steps=tr['eval_steps'],
    save_strategy='steps',
    save_steps=tr['save_steps'],
    save_total_limit=tr['save_total_limit'],
    load_best_model_at_end=True,
    metric_for_best_model=tr['metric_for_best_model'],
    greater_is_better=tr['greater_is_better'],
    fp16=tr['fp16'],
    bf16=tr['bf16'],
    report_to=['tensorboard'],
    seed=tr['seed'],
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CFG['early_stopping']['patience'])],
)

trainer.train()
trainer.save_model(str(OUT))

## 7. 回写 adapter 到 Drive / Kaggle Output

(Colab 里 OUT 已经在 Drive 下,这步可跳过。Kaggle 的 OUT 在 `/kaggle/working`,会自动纳入 Notebook Output。)

In [ ]:
import shutil
print('adapter files in', OUT, ':')
for f in sorted(OUT.glob('**/adapter*')):
    print('  ', f.relative_to(OUT), '(', f.stat().st_size // 1024, 'KB )')

if IS_KAGGLE:
    print('\n在 Kaggle Notebook 右侧 Output 面板下载 artifacts/models/qwen_lora_csrc_1_5b/')
elif IS_COLAB:
    print('\nadapter 已经写在 Google Drive 的', OUT, ', git pull 到本地后即可使用')


## 8. Smoke-test 推理 (在 1.5B adapter 上)

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
infer = PeftModel.from_pretrained(base, str(OUT))
infer.eval()

messages = [
    {'role': 'system', 'content': SYS},
    {'role': 'user', 'content': '2022 年证监会查处的董事长因内幕交易被罚款的案件有哪些? 请引用 [EventID=xxx] 格式。'},
]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors='pt').to(infer.device)
with torch.no_grad():
    out = infer.generate(
        **inputs,
        max_new_tokens=CFG['inference']['max_new_tokens'],
        temperature=CFG['inference']['temperature'],
        top_p=CFG['inference']['top_p'],
        repetition_penalty=CFG['inference']['repetition_penalty'],
        do_sample=True,
    )
print(tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

## 9. 回程

- 把 `artifacts/models/qwen_lora_csrc_1_5b/` pull 回本地仓库
- 修改 `scripts/evaluate_generation_m4_4.py` 里 `LORA_DIR` 常量指向新 adapter + `BASE_MODEL='Qwen/Qwen2.5-1.5B-Instruct'`
- 重跑 `python scripts/evaluate_generation_m4_4.py --n-samples 30`
- 把新的 G3 指标追加到论文 §4.2 的对照表

### 期待的 1.5B 提升
- **EID 命中率**: 0.5B 的 20.0% → 预计 1.5B 35-45%(受限于检索器 R@5=0.388 的天花板)
- **格式合规率**: 0.5B 的 76.7% → 预计 1.5B **85-95%**
- **幻觉数字率**: 0.5B 的 3.3% → 预计 1.5B **≤ 2%**
- **延迟**: 0.5B 的 10.5s → 预计 1.5B 约 18-22s (T4 上本地 inference)

> 论文 §7 局限部分已注明:本项目主交付为 0.5B (本机 2060S 8 GB 唯一可行路径),1.5B 作为**可复现的 Colab 方案**列出。